# 5 vs 18: Did we really grow 18% in H1 2026?

**Problem:** The plan for H1 2026 said "+18% growth over H1 2025". When we measure what actually happened, growth is only about **+5%**. Where does the gap come from, and which cities and months caused it?

**Steps we will follow**

1. Setup: settings and data
2. Count tests per booking
3. City table: H1-25 vs H1-26 vs plan
4. Put a rupee value on the gap
5. Monthly attainment
6. Channel view
7. Does the plan really equal +18%?
8. Insights

**Key idea:** the plan was built on **booked** tests, not invoiced tests. So we compare plan vs actual on booked tests (like with like). Only at the end do we convert the gap to revenue.

## 1. Setup

### Check that the earlier cells have been run

This notebook continues from the first cell of the project, which defines the helper functions and constants we reuse.

- `read()` loads one table by name
- `clean_bookings()` cleans the bookings table
- `main()` builds the invoiced fact table `f`
- `CRORE` = 1 crore = 10,000,000, to show rupees in crores

**What we used:** a list comprehension and `globals()` (the dictionary of everything defined so far).

In [41]:
import os
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.3f}".format
pd.options.display.width = 200
pd.options.display.max_columns = 30

DATA_DIR = '/content/drive/MyDrive/merged'  # Change this to your actual data path if it's not the current folder
CUR = ("2026-01-01", "2026-06-30")   # current period
PRV = ("2025-01-01", "2025-06-30")   # prior period
CRORE = 1e7

# 1. LOAD
def read(name, **kw):
    return pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv.gz"), **kw)

def norm_id(s: pd.Series) -> pd.Series:
    return s.astype("string").str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)

# 2. CLEAN
def clean_bookings(b: pd.DataFrame) -> pd.DataFrame:
    b = b.drop_duplicates("booking_id").copy()
    b["booking_id"] = b["booking_id"].str.strip()
    for c in ("created_at", "slot_at"):
        b[c] = pd.to_datetime(b[c], errors="coerce")

    cancelled = b["status_at_export"].astype("string").str.lower().str.contains("cancel", na=False)
    superseded = b["booking_id"].isin(b["rescheduled_from"].dropna())
    return b[~cancelled & ~superseded]

def clean_lines(l: pd.DataFrame, valid_ids: pd.Series) -> pd.DataFrame:
    return l[l["booking_id"].isin(valid_ids)].copy()

def clean_invoices(inv: pd.DataFrame) -> pd.DataFrame:
    inv = inv.drop_duplicates("invoice_id").copy()
    inv["invoice_date"] = pd.to_datetime(inv["invoice_date"], errors="coerce")
    inv["amount"] = pd.to_numeric(inv["amount"], errors="coerce")
    inv["invoice_key"] = norm_id(inv["invoice_id"])
    bad = inv["invoice_date"].isna() | inv["amount"].isna()
    return inv[~bad]

# 3. FACT TABLE
def build_fact(inv, bookings, lines) -> pd.DataFrame:
    samples = lines.groupby("booking_id").size().rename("samples")
    f = inv.merge(bookings[["booking_id", "source_system", "tests_total", "home_fee"]],
                  on="booking_id", how="inner")
    f["samples"] = f["booking_id"].map(samples).fillna(0)
    f["period"] = np.select(
        [f["invoice_date"].between(*CUR), f["invoice_date"].between(*PRV)], ["CUR", "PRV"], default="OUT")
    f["month"] = f["invoice_date"].dt.to_period("M").astype(str)
    return f

# 4. MAIN AGGREGATION WRAPPER
def main():
    bookings = clean_bookings(read("bookings"))
    lines = clean_lines(read("booking_lines", usecols=["booking_id", "test_code", "test_price", "list_price_rs"]),
                        bookings["booking_id"])
    inv = clean_invoices(read("invoices"))
    f = build_fact(inv, bookings, lines)
    # Return dummy objects for s and d since the notebook only cares about 'f' for now
    return f, None, None

In [42]:
needed = ["read", "clean_bookings", "main", "CRORE"]
missing = [name for name in needed if name not in globals()]
print("Missing:", missing if missing else "nothing, good to go")

Missing: nothing, good to go


### Settings in one place

We keep all the "knobs" at the top so we never hard-code them later.

- `BASIS_DATE`: which date decides the month of a booking. `slot_at` = service date (gives about +5.0%), `created_at` = date the booking was made (gives about +3.5%)
- `CLAIMED_GROWTH`: the +18% the plan claims
- `PRV_M` / `CUR_M`: the six months of H1-25 and H1-26 as text like `"2026-01"`

**What we used:** constants, a list comprehension, `range(1, 7)` (gives 1 to 6) and an f-string with `:02d` (pads the month to two digits, so `1` becomes `01`).

In [43]:
import pandas as pd

BASIS_DATE = "slot_at"
CLAIMED_GROWTH = 0.18

PRV_M = [f"2025-{m:02d}" for m in range(1, 7)]
CUR_M = [f"2026-{m:02d}" for m in range(1, 7)]

print(PRV_M)
print(CUR_M)

['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06']
['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']


## 2. Load the data

### Invoiced fact table `f`

`f` has one row per invoiced item, with `period` (`PRV` or `CUR`), `samples` (tests invoiced) and `amount` (revenue). We only call `main()` if `f` is not already in memory, because it can be slow.

**What we used:** an `if` check on `globals()`, `getattr()` to safely look at the columns, and tuple unpacking (`f, s, d = main()`).

In [44]:
if "f" not in globals() or "samples" not in getattr(f, "columns", []):
    f, s, d = main()

f.head()

,invoice_id,invoice_date,booking_id,patient_id,city,channel,amount,source_file,invoice_key,source_system,tests_total,home_fee,samples,period,month
0,INV-2024-000001,2024-12-30,BK-BLR-202501-006763,PBLR0044005,Bengaluru,clinic,280.000,datapack/invoices/invoices_2024Q4.csv.gz,INV2024000001,legacy,280.000,0,2,OUT,2024-12
1,INV-2024-000002,2024-12-30,BK-BLR-202501-007103,PBLR0044345,Bengaluru,clinic,"1,070.000",datapack/invoices/invoices_2024Q4.csv.gz,INV2024000002,legacy,"1,070.000",0,3,OUT,2024-12
2,INV-2024-000003,2024-12-31,BK-PUN-202501-001985,PPUN0008860,Pune,clinic,350.000,datapack/invoices/invoices_2024Q4.csv.gz,INV2024000003,legacy,350.000,0,1,OUT,2024-12
3,INV-2024-000004,2024-12-31,BK-CHE-202501-001176,PCHE0012203,Chennai,home,"3,299.000",datapack/invoices/invoices_2024Q4.csv.gz,INV2024000004,legacy,"3,149.000",150,2,OUT,2024-12
4,INV-2025-000001,2025-01-01,BK-AHM-202501-005119,PAHM0034226,Ahmedabad,clinic,"2,649.000",datapack/invoices/invoices_2025Q1.csv.gz,INV2025000001,legacy,"2,649.000",0,4,PRV,2025-01


### The plan table

`plan_long` has one row per city per month with a `target` (planned tests). We look at the months to make sure they are written the same way as `CUR_M`, otherwise our filters later will silently return nothing.

**What we used:** `read()`, `.head()`, `.unique()`.

In [45]:
plan = read("plan_long")
display(plan.head())
plan["month"].unique()

,city,metric,month,target
0,Ahmedabad,test_volume_target_2026,2026-01,15119
1,Bengaluru,test_volume_target_2026,2026-01,20380
2,Chennai,test_volume_target_2026,2026-01,15474
3,Hyderabad,test_volume_target_2026,2026-01,14840
4,Jaipur,test_volume_target_2026,2026-01,8853


array(['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06'],
      dtype=object)

### The bookings table

We load only the columns we need (`usecols`), then clean it.

- `slot_at` = the service date
- `created_at` = when the booking was made
- `rescheduled_from`, `status_at_export` = used inside `clean_bookings()` to drop cancelled or duplicate bookings

**What we used:** `read(..., usecols=[...])` (faster, less memory) and a function call that wraps another function call.

In [46]:
bk = clean_bookings(read("bookings", usecols=["booking_id", "city", "channel", "created_at", "slot_at",
                                              "status_at_export", "rescheduled_from"]))
bk.head()

/tmp/ipykernel_1346/1718738606.py:16: DtypeWarning: Columns (11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv.gz"), **kw)


,booking_id,created_at,slot_at,city,channel,rescheduled_from,status_at_export
0,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,clinic,NaN,NaN
1,BK-AHM-202501-000002,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,clinic,NaN,NaN
2,BK-AHM-202501-000003,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,home,NaN,NaN
3,BK-AHM-202501-000004,2025-01-20 02:48:00,2025-01-24 07:15:00,Ahmedabad,clinic,NaN,NaN
4,BK-AHM-202501-000005,2025-01-06 08:21:00,2025-01-11 09:56:00,Ahmedabad,clinic,NaN,NaN


## 3. Count tests per booking

One booking can have several tests, and the plan is in tests, not bookings. So we count the rows per `booking_id` in `booking_lines`, then attach that count to each booking.

- `groupby("booking_id").size()` gives a Series: booking_id to number of lines
- `.map(...)` looks up each booking_id in that Series
- `.fillna(0)` gives 0 tests to bookings that have no lines

**What we used:** `groupby().size()`, `map()`, `fillna()`.

In [47]:
tests_per_booking = read("booking_lines", usecols=["booking_id"]).groupby("booking_id").size()

bk["tests"] = bk["booking_id"].map(tests_per_booking).fillna(0)
bk[["booking_id", "tests"]].head()

,booking_id,tests
0,BK-AHM-202501-000001,3
1,BK-AHM-202501-000002,1
2,BK-AHM-202501-000003,4
3,BK-AHM-202501-000004,1
4,BK-AHM-202501-000005,2


### Month column and the two periods

- `.dt.to_period("M")` turns a date into a month (2026-03-14 becomes 2026-03)
- `.astype(str)` makes it text so it matches `PRV_M` / `CUR_M`
- `.isin(list)` keeps rows whose month is in the list

We end with three tables: `b25` (bookings in H1-25), `b26` (bookings in H1-26) and `f26` (invoiced rows of H1-26).

**What we used:** `.dt` accessor, `to_period`, `isin`, boolean filtering.

In [48]:
bk["month"] = bk[BASIS_DATE].dt.to_period("M").astype(str)

b25 = bk[bk["month"].isin(PRV_M)]
b26 = bk[bk["month"].isin(CUR_M)]
f26 = f[f["period"] == "CUR"]

print("H1-25 booked tests:", int(b25["tests"].sum()))
print("H1-26 booked tests:", int(b26["tests"].sum()))

H1-25 booked tests: 436302
H1-26 booked tests: 458002


## 4. City table: booked tests vs plan

### Step 1: the three raw columns

For each city we want last year's booked tests, this year's booked tests and this year's plan. Each is a `groupby("city")` then `sum()`, and we put them side by side in one DataFrame (pandas lines them up by the city name).

**What we used:** `groupby().sum()`, `pd.DataFrame({...})`.

In [49]:
city = pd.DataFrame({
    "H1-25 booked": b25.groupby("city")["tests"].sum(),
    "H1-26 booked": b26.groupby("city")["tests"].sum(),
    "H1-26 plan":   plan[plan["month"].isin(CUR_M)].groupby("city")["target"].sum(),
})
city

,H1-25 booked,H1-26 booked,H1-26 plan
city,,,
Ahmedabad,76926,85084,90773
Bengaluru,100070,107661,120085
Chennai,80267,80772,93912
Hyderabad,73237,78674,86421
Jaipur,45149,40236,52372
Pune,60653,65575,72177


### Step 2: add a TOTAL row

We add the TOTAL row **before** the ratio columns. A total growth rate must be (total this year / total last year) - 1. Averaging the city growth rates would give the wrong answer, because big cities and small cities would count equally.

**What we used:** `.loc["TOTAL"] = ...` (adds a row) and `.sum()` (column totals).

In [50]:
city.loc["TOTAL"] = city.sum()
city

,H1-25 booked,H1-26 booked,H1-26 plan
city,,,
Ahmedabad,76926,85084,90773
Bengaluru,100070,107661,120085
Chennai,80267,80772,93912
Hyderabad,73237,78674,86421
Jaipur,45149,40236,52372
Pune,60653,65575,72177
TOTAL,436302,458002,515740


### Step 3: the ratio columns

| Column | Formula | Meaning |
|---|---|---|
| Actual growth | H1-26 booked / H1-25 booked - 1 | what really happened |
| Plan growth (implied) | H1-26 plan / H1-25 booked - 1 | what the plan asked for |
| Attainment | H1-26 booked / H1-26 plan | how much of the plan we achieved |
| Gap (tests) | plan - booked | tests we are short by |
| Growth gap (pp) | (actual growth - plan growth) x 100 | gap in percentage points |

**What we used:** column arithmetic (pandas divides whole columns at once, row by row).

In [51]:
city["Actual growth"]         = city["H1-26 booked"] / city["H1-25 booked"] - 1
city["Plan growth (implied)"] = city["H1-26 plan"] / city["H1-25 booked"] - 1
city["Attainment"]            = city["H1-26 booked"] / city["H1-26 plan"]
city["Gap (tests)"]           = city["H1-26 plan"] - city["H1-26 booked"]
city["Growth gap (pp)"]       = (city["Actual growth"] - city["Plan growth (implied)"]) * 100

city.round(3)

,H1-25 booked,H1-26 booked,H1-26 plan,Actual growth,Plan growth (implied),Attainment,Gap (tests),Growth gap (pp)
city,,,,,,,,
Ahmedabad,76926,85084,90773,0.106,0.180,0.937,5689,-7.395
Bengaluru,100070,107661,120085,0.076,0.200,0.897,12424,-12.415
Chennai,80267,80772,93912,0.006,0.170,0.860,13140,-16.370
Hyderabad,73237,78674,86421,0.074,0.180,0.910,7747,-10.578
Jaipur,45149,40236,52372,-0.109,0.160,0.768,12136,-26.880
Pune,60653,65575,72177,0.081,0.190,0.909,6602,-10.885
TOTAL,436302,458002,515740,0.050,0.182,0.888,57738,-13.233


## 5. What is the gap worth in rupees?

The gap is in booked tests, but revenue comes from invoiced tests. So we convert in two steps:

1. **Conversion** = invoiced tests / booked tests (not every booked test gets invoiced)
2. **Revenue per invoiced test** = total amount / total invoiced tests

Revenue gap = gap in tests x conversion x revenue per test, divided by `CRORE`.

This is **indicative**, because it assumes the missing tests would have converted and earned like the average test.

**What we used:** `.sum()` on columns of two different tables, and a formula applied to a whole column.

In [52]:
conv = f26["samples"].sum() / b26["tests"].sum()
rev_per_test = f26["amount"].sum() / f26["samples"].sum()

print(f"Booked -> invoiced conversion: {conv:.1%}")
print(f"Revenue per invoiced test: INR {rev_per_test:,.0f}")

Booked -> invoiced conversion: 82.8%
Revenue per invoiced test: INR 701


In [53]:
city["Revenue gap (INR cr, indicative)"] = city["Gap (tests)"] * conv * rev_per_test / CRORE

print(f"=== PLAN vs ACTUAL, H1 2026 - BOOKED TESTS by {BASIS_DATE} ===")
print(city.to_string(float_format=lambda x: f"{x:,.3f}"))

=== PLAN vs ACTUAL, H1 2026 - BOOKED TESTS by slot_at ===
           H1-25 booked  H1-26 booked  H1-26 plan  Actual growth  Plan growth (implied)  Attainment  Gap (tests)  Growth gap (pp)  Revenue gap (INR cr, indicative)
city                                                                                                                                                               
Ahmedabad         76926         85084       90773          0.106                  0.180       0.937         5689           -7.395                             0.331
Bengaluru        100070        107661      120085          0.076                  0.200       0.897        12424          -12.415                             0.722
Chennai           80267         80772       93912          0.006                  0.170       0.860        13140          -16.370                             0.763
Hyderabad         73237         78674       86421          0.074                  0.180       0.910         7747          

## 6. Monthly attainment

Is the shortfall steady, or did it get better or worse through the half-year?

### Actual per city per month, next to the plan

- `groupby(["city", "month"])` groups by two columns, giving a two-level index
- `pd.concat([...], axis=1)` places two Series side by side, matched on that index
- If a city/month exists in only one table we get NaN, so we print a warning instead of failing silently

**What we used:** two-column `groupby`, `set_index`, `pd.concat(axis=1)`, `.isna().any()`.

In [54]:
act_m = b26.groupby(["city", "month"])["tests"].sum().rename("actual")

pm = pd.concat([plan.set_index(["city", "month"])["target"], act_m], axis=1)

if pm.isna().any().any():
    print("[warn] city/month keys present in only one of plan / bookings:")
    print(pm[pm.isna().any(axis=1)])
else:
    print("Plan and bookings line up perfectly.")

Plan and bookings line up perfectly.


### Attainment table

`unstack("month")` turns the month level of the index into columns, so we get a cities x months grid, easy to scan for the weak spots.

**What we used:** `unstack()`, `.round()`.

In [55]:
pm["attainment"] = pm["actual"] / pm["target"]

pm["attainment"].unstack("month").round(3)

month,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
city,,,,,,
Ahmedabad,0.926,0.982,0.970,0.893,0.930,0.926
Bengaluru,0.891,0.986,0.830,0.920,0.895,0.861
Chennai,0.885,0.977,0.807,0.828,0.848,0.823
Hyderabad,0.889,0.919,0.933,0.885,0.884,0.955
Jaipur,0.764,0.726,0.720,0.803,0.768,0.846
Pune,0.932,0.878,0.978,0.907,0.841,0.916


### Monthly totals across all cities

**What we used:** `groupby("month")` on one level of the index, and a list of columns `[["target", "actual"]]`.

In [56]:
mt = pm.groupby("month")[["target", "actual"]].sum()
mt["attainment"] = mt["actual"] / mt["target"]
mt

,target,actual,attainment
month,,,
2026-01,87002,77321,0.889
2026-02,86293,80123,0.928
2026-03,87882,76770,0.874
2026-04,85616,75287,0.879
2026-05,84678,73717,0.871
2026-06,84269,74784,0.887


## 7. Channel view

The plan is only at city level, so for channels we cannot compute attainment. We can still see growth and share of volume.

**What we used:** `groupby("channel")`, growth formula, share = value / total of the column.

In [57]:
ch = pd.DataFrame({
    "H1-25": b25.groupby("channel")["tests"].sum(),
    "H1-26": b26.groupby("channel")["tests"].sum(),
})
ch["Growth"] = ch["H1-26"] / ch["H1-25"] - 1
ch["Share of H1-26"] = ch["H1-26"] / ch["H1-26"].sum()

print("=== BOOKED TESTS BY CHANNEL (no channel-level plan) ===")
print(ch.to_string(float_format=lambda x: f"{x:,.3f}"))

=== BOOKED TESTS BY CHANNEL (no channel-level plan) ===
          H1-25   H1-26  Growth  Share of H1-26
channel                                        
clinic   287186  253121  -0.119           0.553
home     149116  204881   0.374           0.447


## 8. Does the plan match the claimed +18%?

We pull the TOTAL row out as `tot` and compare:

- plan-implied growth vs `CLAIMED_GROWTH` (within 1 percentage point counts as a match)
- actual growth vs plan

**What we used:** `city.loc["TOTAL"]` (select one row), `abs()`, f-string formats: `:+.1%` shows a sign and one decimal as a percent.

In [58]:
tot = city.loc["TOTAL"]

matches = abs(tot["Plan growth (implied)"] - CLAIMED_GROWTH) <= 0.01

print(f"=== CHECK vs claimed +{CLAIMED_GROWTH:.0%} plan ===")
print(f"Plan-implied growth : {tot['Plan growth (implied)']:+.1%}  (matches within 1pp: {matches})")
print(f"Actual growth       : {tot['Actual growth']:+.1%}  (gap {tot['Growth gap (pp)']:+.1f} pp)")

=== CHECK vs claimed +18% plan ===
Plan-implied growth : +18.2%  (matches within 1pp: True)
Actual growth       : +5.0%  (gap -13.2 pp)


## 9. Insights

We first prepare the small pieces, and then print the sentences.

### Pieces

- `c` = cities only (drop the TOTAL row)
- `gap_share` = each city's share of the total shortfall, biggest first
- `top2` = the two cities contributing most
- `neg` = cities whose volume fell year on year
- `trend` = monthly attainment (Jan to Jun)

**What we used:** `.drop()`, `.sort_values()`, `.head()`, boolean filter with `.index.tolist()`.

In [59]:
c = city.drop("TOTAL")

gap_share = (c["Gap (tests)"] / c["Gap (tests)"].sum()).sort_values(ascending=False)
top2 = gap_share.head(2)

neg = c[c["Actual growth"] < 0].index.tolist()
trend = mt["attainment"]

print(top2)
print("Cities that shrank:", neg)

city
Chennai     0.228
Bengaluru   0.215
Name: Gap (tests), dtype: float64
Cities that shrank: ['Jaipur']


### Headline insights

`idxmin()` / `idxmax()` return the **name** of the row with the smallest / largest value, which is how we find the weakest and strongest city.

**What we used:** `idxmin()`, `idxmax()`, `iloc[0]` / `iloc[-1]` (first / last item), a conditional `if`.

In [60]:
print("=== INSIGHTS ===")
print(f"- Booked tests grew {tot['Actual growth']:+.1%} vs a plan of {tot['Plan growth (implied)']:+.1%}: "
      f"{tot['Attainment']:.0%} of plan, short by {tot['Gap (tests)']:,.0f} tests "
      f"(~INR {tot['Revenue gap (INR cr, indicative)']:.1f} cr).")

print(f"- Weakest: {c['Attainment'].idxmin()} ({c['Attainment'].min():.0%} of plan); "
      f"strongest: {c['Attainment'].idxmax()} ({c['Attainment'].max():.0%}).")

print(f"- {top2.index[0]} and {top2.index[1]} account for {top2.sum():.0%} of the test shortfall.")

if neg:
    print(f"- Volume shrank year-on-year in: {', '.join(neg)}.")

=== INSIGHTS ===
- Booked tests grew +5.0% vs a plan of +18.2%: 89% of plan, short by 57,738 tests (~INR 3.4 cr).
- Weakest: Jaipur (77% of plan); strongest: Ahmedabad (94%).
- Chennai and Bengaluru account for 44% of the test shortfall.
- Volume shrank year-on-year in: Jaipur.


### Trend and revenue vs volume

- **Trend:** compare June attainment with January. Within 1 point = flat.
- **Revenue vs volume:** if invoiced revenue grew faster than booked tests, then revenue per test (realisation: price and mix) carried the growth, not volume.

**What we used:** nested conditional expression (`'flat' if ... else 'improving' if ... else 'declining'`), filtering `f` by `period`.

In [61]:
change = trend.iloc[-1] - trend.iloc[0]
direction = "flat" if abs(change) < 0.01 else "improving" if change > 0 else "declining"
print(f"- Attainment Jan->Jun: {trend.iloc[0]:.0%} -> {trend.iloc[-1]:.0%} ({direction}).")

rev_g = f26["amount"].sum() / f[f["period"] == "PRV"]["amount"].sum() - 1

if rev_g > tot["Actual growth"]:
    verdict = "revenue outgrew volume, so realisation (revenue per test) carried the growth."
else:
    verdict = "revenue did not outgrow volume."

print(f"- Invoiced revenue {rev_g:+.1%} vs booked test volume {tot['Actual growth']:+.1%}: {verdict}")

- Attainment Jan->Jun: 89% -> 89% (flat).
- Invoiced revenue +11.1% vs booked test volume +5.0%: revenue outgrew volume, so realisation (revenue per test) carried the growth.


## Recap: what we used, in one table

| Tool | Where we used it |
|---|---|
| `list comprehension`, `f-string` | building month lists, printing results |
| `groupby().size()`, `map()`, `fillna()` | tests per booking |
| `.dt.to_period("M")`, `isin()` | month column, H1-25 / H1-26 split |
| `groupby().sum()`, `pd.DataFrame({...})` | city and channel tables |
| `.loc["TOTAL"] = .sum()` | total row (before ratios) |
| column arithmetic | growth, attainment, gap, revenue gap |
| `pd.concat(axis=1)`, `unstack()` | plan vs actual by month |
| `idxmin()`, `idxmax()`, `sort_values()`, `drop()` | finding weakest city and top shortfall |

**Answer to "5 vs 18":** the plan asked for about +18% growth on booked tests, we delivered about +5%. The shortfall is concentrated in a few cities (see the top-2 line), and revenue growth leans on realisation more than volume (see the last line).